# Video2X Kaggle notebook (fork)

Upscales an anime episode to **1080p** with Real-ESRGAN (`realesr-animevideov3`, x2, then a Lanczos resize to 1080p), using **both T4 GPUs** of a Kaggle "GPU T4 x2" session.

Video2X runs on one GPU per process, so this notebook cuts the episode into chunks at keyframes, runs one `video2x` process per GPU (each takes the next free chunk), and joins the results with the original audio and subtitles. Expect close to 2x the speed of one T4.

**Before you start (Kaggle sidebar, Settings):**
1. Accelerator: **GPU T4 x2**.
2. Internet: **On** (needed to clone and build; Kaggle asks for a phone-verified account).
3. Add your episode as a private **Dataset** (Add Input > Upload). It appears under `/kaggle/input/...`.

**Running it:** run the cells in order, or use *Save Version > Save & Run All* to run the whole thing unattended (a session lasts up to 12 hours). The result is written to `/kaggle/working/output`, which Kaggle keeps as the notebook's Output. Finished chunks are kept in `/kaggle/working/v2x_*`, so an interrupted run resumes from the last finished chunk if the same working files are still there.

**Speed test (the default settings):** cell 2.1 is set to process the first 10 seconds as four chunks of about 75 frames, so each GPU gets two. It prints the overall fps at the end. To compare with one GPU, set `devices = [0]`. For a whole episode set `limit_seconds = 0`, `precise_split = False` and `chunk_seconds` to 120-240; leave `remove_pulldown = True` if the source has 3:2 pulldown (the split step then re-encodes the video once, near-losslessly).

## 1. Setup

In [ ]:
# 1.1 GPU and Vulkan check
#
# Video2X uses Vulkan, not CUDA. Kaggle's containers do not always ship the NVIDIA Vulkan driver,
# so this cell checks for it and installs the user-space part if it is missing.

import os
import re
import subprocess

os.makedirs("/kaggle/temp", exist_ok=True)


def sh(cmd, show=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if show and output:
        print(output)
    return result.returncode, output


code, output = sh("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader")
if code != 0:
    raise RuntimeError("No NVIDIA GPU found. In the Kaggle sidebar open Settings > Accelerator, choose GPU T4 x2, and run again.")
gpu_rows = [[part.strip() for part in line.split(",")] for line in output.splitlines()]
driver = gpu_rows[0][2]
print(f"{len(gpu_rows)} GPU(s), driver {driver}")

sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools", show=False)

ICD = "/usr/share/vulkan/icd.d/nvidia_icd.json"


def ensure_icd():
    if not os.path.exists(ICD):
        os.makedirs(os.path.dirname(ICD), exist_ok=True)
        with open(ICD, "w") as handle:
            handle.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.3.0"}}\n')
    os.environ["VK_ICD_FILENAMES"] = ICD


def vulkan_gpus():
    ensure_icd()
    sh("ldconfig", show=False)
    _, text = sh("vulkaninfo --summary", show=False)
    names = re.findall(r"deviceName\s*=\s*(.+)", text)
    return [name.strip() for name in names if "llvmpipe" not in name.lower()]


found = vulkan_gpus()
if not found:
    major = driver.split(".")[0]
    print(f"No Vulkan GPU yet. Trying the Ubuntu package libnvidia-gl-{major} ...")
    sh(f"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libnvidia-gl-{major}", show=False)
    found = vulkan_gpus()
if not found:
    print(f"Still none. Installing the user-space libraries of driver {driver} from NVIDIA (no kernel module) ...")
    url = f"https://download.nvidia.com/XFree86/Linux-x86_64/{driver}/NVIDIA-Linux-x86_64-{driver}.run"
    sh(f"cd /kaggle/temp && wget -q -O nvidia.run {url} && sh nvidia.run --silent --ui=none --no-kernel-module "
       "--no-nvidia-modprobe --no-questions --no-x-check --no-backup --no-install-compat32-libs")
    found = vulkan_gpus()

print("Vulkan GPUs:", found or "none")
if not found:
    raise RuntimeError(
        "Vulkan cannot see the NVIDIA GPUs in this Kaggle session, so Video2X cannot run here. "
        "Copy the output above and report it; the alternative is a CUDA-based backend."
    )
if len(found) < len(gpu_rows):
    print(f"Warning: nvidia-smi shows {len(gpu_rows)} GPUs but Vulkan shows {len(found)}. Only those will be used.")
else:
    print("OK: every GPU is visible to Vulkan.")

In [ ]:
# 1.2 Build Video2X from source
#
# - The first build takes roughly 10-20 minutes. You only need to run it once per session.
# - Fork submodules (ncnn, spdlog and the model wrappers) are fetched automatically; the large
#   Boost submodule is not needed because the system Boost is used.
# - The sources and the build live in /kaggle/temp, which is not kept as notebook output.

repo_url = "https://github.com/adeshboudhnicedigitals/video2x.git"
branch = "master"

!add-apt-repository -y ppa:ubuntuhandbook1/ffmpeg7
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends ffmpeg git build-essential cmake ninja-build libavcodec-dev libavdevice-dev libavfilter-dev libavformat-dev libavutil-dev libswscale-dev libvulkan-dev glslang-tools libomp-dev libboost-program-options-dev

!rm -rf /kaggle/temp/video2x-src
!git clone --branch {branch} {repo_url} /kaggle/temp/video2x-src
!cd /kaggle/temp/video2x-src && git submodule update --init --recursive -- third_party/ncnn third_party/spdlog third_party/librealesrgan_ncnn_vulkan third_party/librealcugan_ncnn_vulkan third_party/librife_ncnn_vulkan

!cd /kaggle/temp/video2x-src && cmake -G Ninja -B build -S . -DVIDEO2X_USE_EXTERNAL_NCNN=OFF -DVIDEO2X_USE_EXTERNAL_SPDLOG=OFF -DCMAKE_CXX_COMPILER=g++ -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr
!cd /kaggle/temp/video2x-src && cmake --build build --config Release --target install --parallel
!ldconfig

# Confirm that the freshly built binary has the options this notebook uses
!video2x --help | grep -E "queue-size|realesrgan-tile-size|--device"
!video2x --list-devices

In [ ]:
# 1.3 Input file
#
# Picks the video to process.
# - Add your episode to this notebook as a Dataset; it shows up under /kaggle/input/.
# - Leave input_path empty to use the only video found, or set it to a path from the list.

import os
import pathlib

input_path = ""

VIDEO_SUFFIXES = {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".wmv", ".ts"}


def find_videos():
    found = []
    for base in ("/kaggle/input", "/kaggle/working"):
        for root, dirs, names in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith(".") and not d.startswith("v2x_") and d != "output"]
            for name in names:
                if pathlib.Path(name).suffix.lower() in VIDEO_SUFFIXES:
                    found.append(pathlib.Path(root, name))
    return sorted(found)


videos = find_videos()
if videos:
    print("Videos found:")
    for video in videos:
        print(f"  {video}")

if input_path.strip():
    input_file = pathlib.Path(input_path.strip())
    if not input_file.is_file():
        raise FileNotFoundError(f"input_path does not point to a file: {input_file}")
elif len(videos) == 1:
    input_file = videos[0]
elif not videos:
    raise FileNotFoundError("No videos found. Add your episode as a Dataset (Add Input), or set input_path.")
else:
    raise ValueError("Several videos found. Set input_path to one of the paths listed above.")

print(f"Selected input file: {input_file}")

## 2. Upscale on all GPUs

In [ ]:
# 2.1 Upscale in chunks, one video2x process per GPU
#
# What this cell does:
# 1. Cuts the input into chunks of about `chunk_seconds` at keyframes (stream copy, no re-encoding,
#    video only).
# 2. Starts one worker per GPU. Each worker takes the next unfinished chunk, runs video2x on it
#    and checks that the output has the same number of frames as the chunk.
# 3. Prints a status line every 30 seconds. Cell 2.2 joins the chunks.
#
# Finished chunks are kept, so running the cell again skips them.

import json
import pathlib
import queue
import re
import subprocess
import threading
import time

if 'input_file' not in globals() or input_file is None:
    raise NameError("Input file not specified. Did you forget to run cell 1.3?")

# --- Test mode ---
# Set to the first 10 seconds of the video for a speed test. 0 processes the whole file.
limit_seconds = 10

# - precise_split: cut at exactly every `chunk_seconds` by re-encoding the video (near-lossless, a
#   little slower). Without it the cuts are stream copies that snap to keyframes, and a short clip
#   may have no keyframe to cut at, so it stays one chunk and only one GPU works. Use it for tests
#   and leave it off for a whole episode.
precise_split = True

# - remove_pulldown: drops the repeated frame that 3:2 pulldown adds to every 5th frame (ffmpeg
#   `decimate=cycle=5`), turning 29.97 fps film-sourced video back into its true 23.976 fps. That
#   is 20% fewer frames to upscale, and the audio stays in sync. Measured on the user's Bleach 271
#   file at full resolution: 98% of blocks of 5 frames contain a near-exact repeat; in about 1% of
#   blocks (cuts and breaks in the cadence) a real frame is dropped. Turn it off for video that is
#   natively 29.97 or 30 fps, or if you see judder. It needs a re-encode of the video, like
#   precise_split, so a full episode takes a few extra minutes in the split step.
remove_pulldown = True
# --- Chunks ---
# Smaller chunks balance the two GPUs better and lose less work if something fails; larger chunks
# have less start-up overhead (a few seconds per chunk). Cuts snap to the next keyframe.
chunk_seconds = 2.5  # test value: 4 chunks of about 75 frames, two per GPU. Use 120-240 for a whole episode

# --- Filter and model ---
filter = "realesrgan"  # one of: realesrgan, realcugan
model = "realesr-animevideov3"  # realesrgan: realesrgan-plus-anime, realesrgan-plus, realesr-animevideov3
scale = 2
output_height = 1080  # each upscaled frame is resized to this height; 0 keeps the full upscaled size
cugan_model = "models-se"  # realcugan: models-se, models-pro, models-nose
cugan_noise = "conservative"  # realcugan: conservative, none, 1, 2, 3

# --- Encoder ---
codec = "libx264"
preset = "slow"
crf = 18
queue_size = 4
tile_size = 400  # realesrgan only; 0 = automatic (200 on a T4)

# --- GPUs ---
# Empty = every discrete GPU that Vulkan reports. Or list Vulkan device indices, for example [0, 1].
devices = []

CUGAN_NOISE = {"conservative": -1, "none": 0, "1": 1, "2": 2, "3": 3}
CUGAN_SUPPORT = {
    "models-se": {2: {-1, 0, 1, 2, 3}, 3: {-1, 0, 3}, 4: {-1, 0, 3}},
    "models-pro": {2: {-1, 0, 3}, 3: {-1, 0, 3}},
    "models-nose": {2: {0}},
}
if filter == "realesrgan":
    supported_scales = {2, 3, 4} if model == "realesr-animevideov3" else {4}
    if scale not in supported_scales:
        raise ValueError(f"{model} supports only {sorted(supported_scales)}x, not {scale}x.")
elif filter == "realcugan":
    supported = CUGAN_SUPPORT[cugan_model]
    if scale not in supported or CUGAN_NOISE[cugan_noise] not in supported[scale]:
        raise ValueError(f"{cugan_model} does not have {scale}x with noise '{cugan_noise}'.")
else:
    raise ValueError("filter must be realesrgan or realcugan")


def run(cmd, **kwargs):
    return subprocess.run(cmd, capture_output=True, text=True, **kwargs)


def probe(path, count=False):
    args = ["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
            "stream=width,height,nb_read_frames" if count else "stream=width,height",
            "-of", "json"]
    if count:
        args.insert(3, "-count_frames")
    info = json.loads(run(args + [str(path)], check=True).stdout)["streams"][0]
    return info


def duration_of(path):
    return float(run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                      "-of", "default=nw=1:nk=1", str(path)], check=True).stdout.strip())


def discrete_devices():
    result = run(["video2x", "--list-devices"])
    found, current = [], None
    for line in (result.stdout + result.stderr).splitlines():
        match = re.match(r"^(\d+)\.\s+(.+)$", line)
        if match:
            current = (int(match.group(1)), match.group(2).strip())
        elif current and "Type:" in line:
            if "Discrete" in line or "Virtual" in line:
                found.append(current)
            current = None
    return found


available = discrete_devices()
if devices:
    chosen = [(index, name) for index, name in available if index in devices]
else:
    chosen = available
if not chosen:
    raise RuntimeError("video2x sees no discrete GPU. Run cell 1.1 and 1.2 first and read their output.")
print("GPUs used:", ", ".join(f"{index}: {name}" for index, name in chosen))

# --- Working folders ---
safe_stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", input_file.stem)[:40]
work = pathlib.Path("/kaggle/working") / (f"v2x_{safe_stem}" + ("_test" if limit_seconds else ""))
src_dir = work / "src"
out_dir = work / "out"
src_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

# --- 1. Split into chunks ---
split_marker = src_dir / "split.done"
reencode_split = precise_split or remove_pulldown
split_settings = json.dumps({"input": str(input_file), "limit": limit_seconds, "chunk": chunk_seconds,
                             "precise": precise_split, "pulldown": remove_pulldown})
if not split_marker.exists() or split_marker.read_text() != split_settings:
    # A different split invalidates the finished chunks, so start from clean folders
    for old in list(src_dir.glob("chunk_*.mkv")) + list(out_dir.glob("chunk_*")):
        old.unlink()
    split_marker.unlink(missing_ok=True)
    print("Splitting into chunks (" + ("re-encoding" + (", removing pulldown" if remove_pulldown else "") if reencode_split else "stream copy") + ")...")
    started = time.time()
    split_cmd = ["ffmpeg", "-y", "-loglevel", "error"]
    if limit_seconds:
        split_cmd += ["-t", str(limit_seconds)]
    split_cmd += ["-i", str(input_file), "-map", "0:v:0"]
    if reencode_split:
        if remove_pulldown:
            split_cmd += ["-vf", "decimate=cycle=5", "-fps_mode", "passthrough"]
        split_cmd += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "10",
                      "-force_key_frames", f"expr:gte(t,n_forced*{chunk_seconds})"]
    else:
        split_cmd += ["-c", "copy"]
    split_cmd += ["-an", "-sn", "-f", "segment", "-segment_time", str(chunk_seconds),
                  "-reset_timestamps", "1", str(src_dir / "chunk_%03d.mkv")]
    result = run(split_cmd)
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("Splitting the input failed")
    split_marker.write_text(split_settings)
    print(f"  done in {time.time() - started:.0f} s")

chunks = sorted(src_dir.glob("chunk_*.mkv"))
if not chunks:
    raise RuntimeError("The split produced no chunks.")
chunk_length = {chunk: duration_of(chunk) for chunk in chunks}
total_seconds = sum(chunk_length.values())
width, height = probe(chunks[0])["width"], probe(chunks[0])["height"]
print(f"{len(chunks)} chunks, {total_seconds / 60:.1f} min of video, input {width}x{height}")


def build_command(device, chunk, destination):
    command = [
        "video2x", "--input", str(chunk), "--output", str(destination),
        "--processor", filter, "--device", str(device),
        "--codec", codec, "--bit-rate", "0", "--log-level", "info", "--no-progress",
        "--queue-size", str(queue_size),
        "-e", f"preset={preset}", "-e", f"crf={crf}",
        "--scaling-factor", str(scale),
    ]
    if filter == "realesrgan":
        command += ["--realesrgan-model", model]
        if tile_size > 0:
            command += ["--realesrgan-tile-size", str(tile_size)]
    else:
        command += ["--realcugan-model", cugan_model, f"--noise-level={CUGAN_NOISE[cugan_noise]}"]
    if output_height > 0:
        command += ["--height", str(output_height)]
    return command


# --- 2. Workers ---
todo = queue.Queue()
for chunk in chunks:
    if not (out_dir / chunk.name).exists():
        todo.put(chunk)
skipped = len(chunks) - todo.qsize()
if skipped:
    print(f"{skipped} chunk(s) already finished; skipping them.")

lock = threading.Lock()
processes = []
state = {"done_frames": 0, "done_seconds": sum(chunk_length[c] for c in chunks if (out_dir / c.name).exists()),
         "failed": [], "current": {}}


def worker(device):
    while True:
        try:
            chunk = todo.get_nowait()
        except queue.Empty:
            return
        part = out_dir / (chunk.stem + ".partial.mkv")
        log = out_dir / (chunk.stem + ".log")
        part.unlink(missing_ok=True)
        with lock:
            state["current"][device] = chunk.name
        try:
            with open(log, "w") as handle:
                process = subprocess.Popen(build_command(device, chunk, part), stdout=handle, stderr=subprocess.STDOUT)
                with lock:
                    processes.append(process)
                code = process.wait()
            expected = int(probe(chunk, count=True)["nb_read_frames"])
            produced = int(probe(part, count=True)["nb_read_frames"]) if part.exists() else -1
            if produced != expected:
                raise RuntimeError(f"{chunk.name}: {produced} frames written, {expected} expected (exit code {code}); see {log}")
            part.rename(out_dir / chunk.name)
            with lock:
                state["done_seconds"] += chunk_length[chunk]
                state["done_frames"] += expected
        except Exception as error:
            with lock:
                state["failed"].append(str(error))
        finally:
            with lock:
                state["current"].pop(device, None)


def gpu_status():
    result = run(["nvidia-smi", "--query-gpu=index,utilization.gpu,power.draw,clocks.sm,temperature.gpu",
                  "--format=csv,noheader,nounits"])
    rows = [[part.strip() for part in line.split(",")] for line in result.stdout.strip().splitlines() if line.strip()]
    return "  ".join(f"GPU{r[0]}: {r[1]}% {r[2]}W {r[3]}MHz {r[4]}C" for r in rows)


threads = [threading.Thread(target=worker, args=(index,), daemon=True) for index, _ in chosen]
started = time.time()
try:
    for thread in threads:
        thread.start()
    last_print = 0.0
    while any(thread.is_alive() for thread in threads):
        time.sleep(5)
        if time.time() - last_print >= 30:
            last_print = time.time()
            with lock:
                done = state["done_seconds"]
                running = dict(state["current"])
            elapsed = time.time() - started
            eta = ""
            if done > 0 and elapsed > 0:
                remaining = (total_seconds - done) / (done / elapsed) if done < total_seconds else 0
                eta = f", about {remaining / 60:.0f} min left"
            finished = sum(1 for c in chunks if (out_dir / c.name).exists())
            print(f"[{elapsed / 60:5.1f} min] {finished}/{len(chunks)} chunks done{eta}; running: {running}")
            print(f"           {gpu_status()}")
        if state["failed"]:
            break
finally:
    if state["failed"] or any(thread.is_alive() for thread in threads):
        for process in processes:
            if process.poll() is None:
                process.terminate()

if state["failed"]:
    raise RuntimeError("A chunk failed:\n" + "\n".join(state["failed"]))
elapsed = time.time() - started
print(f"All {len(chunks)} chunks finished in {elapsed:.0f} s ({elapsed / 60:.1f} min), "
      f"{state['done_frames']} frames processed this run: {state['done_frames'] / elapsed:.2f} fps on {len(chosen)} GPU(s).")
print("For comparison: one T4 reached about 1.3 fps on a long run. Start-up per chunk makes short tests look slower.")

In [ ]:
# 2.2 Join the chunks and add the original audio and subtitles
#
# Run this after cell 2.1 has finished all chunks. The result goes to /kaggle/working/output, which
# Kaggle keeps as the notebook's Output (20 GB limit).

import pathlib
import subprocess

if 'chunks' not in globals():
    raise NameError("Run cell 2.1 first.")

output_dir = pathlib.Path("/kaggle/working/output")
output_dir.mkdir(parents=True, exist_ok=True)
container = ".mkv"  # mkv keeps most subtitle formats; mp4 may drop them

finished = [out_dir / chunk.name for chunk in chunks]
missing = [path.name for path in finished if not path.exists()]
if missing:
    raise RuntimeError(f"Chunks not finished yet: {missing}")

list_file = work / "concat.txt"
list_file.write_text("".join(f"file '{path}'\n" for path in finished))

final = output_dir / f"{input_file.stem}.{output_height or 'upscaled'}p{'.test' if limit_seconds else ''}{container}"
final.unlink(missing_ok=True)


def mux(with_subtitles):
    command = ["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0", "-i", str(list_file),
               "-i", str(input_file), "-map", "0:v:0", "-map", "1:a?"]
    if with_subtitles:
        command += ["-map", "1:s?"]
    command += ["-c", "copy"]
    if limit_seconds:
        command += ["-t", str(limit_seconds)]
    return subprocess.run(command + [str(final)], capture_output=True, text=True)


result = mux(True)
if result.returncode != 0:
    print("Copying the subtitles failed; retrying without subtitles.")
    final.unlink(missing_ok=True)
    result = mux(False)
if result.returncode != 0:
    print(result.stderr[-1500:])
    raise RuntimeError("Joining the chunks failed")

expected = min(duration_of(input_file), limit_seconds) if limit_seconds else duration_of(input_file)
actual = duration_of(final)
size = final.stat().st_size / 1e6
info = probe(final)
print(f"Written: {final}")
print(f"  {info['width']}x{info['height']}, {size:.0f} MB, {actual:.1f} s (input {expected:.1f} s)")
if abs(actual - expected) > 1.0:
    print("Warning: the duration differs from the input by more than a second. Check the output.")